
# Project B | 주간 수요 기준 예측 모델 구축

## 분석 목적

상품군별 주간 주문 수량에 대해 단순 예측 모델을 구축하고, 이후 통계적 시계열 모델(ETS·ARIMA)의 성능을 평가하기 위한 기준점을 마련한다.

### 데이터 및 예측 설정
- 분석 대상: 상위 5개 상품군
- 전체 관측 기간: 76주
- 학습·모델 검증 구간: 최초 64주
- 최종 평가 구간: 마지막 12주 (모델 선택에 사용하지 않음)
- 예측 목표: 향후 1~4주 주문 상품 수량

### 비교 모델

**1. Naive Forecast**

예측 시점에 마지막으로 관측한 주문 수량을 미래에도 그대로 유지한다고 가정한다.

$$
\hat{Y}_{t+h|t}=Y_t,\quad h=1,2,3,4
$$

**2. Moving Average Forecast**

예측 시점까지 관측한 최근 4주의 평균 주문 수량을 향후 1~4주 예측값으로 사용한다.

### 평가 방법

학습 구간 내부에서 Rolling-origin 검증을 수행한다.

- MAE: 평균 절대오차
- WAPE: 전체 실제 수량 대비 절대오차 비율
- Horizon: 1~4주 예측 시차별 성능 비교

### 검증 원칙
- 예측 시점 이후의 실제 주문 수량을 입력에 사용하지 않는다.
- 모든 모델에 동일한 검증 구간과 평가 지표를 적용한다.
- 최종 평가 데이터는 모델 선정 이후에만 사용한다.


In [1]:
import sys
print(sys.version)


3.12.14 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:37:13) [MSC v.1942 64 bit (AMD64)]


In [8]:

# 데이터 처리
import pandas as pd
import numpy as np

# 시각화
import matplotlib.pyplot as plt

# PostgreSQL 연결
import os
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from dotenv import load_dotenv, find_dotenv

# 예측 성능 평가
from sklearn.metrics import mean_absolute_error

In [10]:
# 1. 환경변수 로드
load_dotenv(find_dotenv(usecwd=True))

# 2. 데이터베이스 접속 정보 구성
db_url = URL.create(
    drivername='postgresql+psycopg',
    username=os.getenv('DB_USER'),
    password=os.getenv('DB_PASSWORD'),
    host=os.getenv('DB_HOST'),
    port=int(os.getenv('DB_PORT', '5432')),
    database=os.getenv('DB_NAME')
)
# 3. SQLAlchemy 엔진 생성
engine = create_engine(db_url, pool_pre_ping=True)

# 4. 연결 후 SQL 실행
with engine.connect() as conn:
    view_count = conn.execute(text('SELECT COUNT(*) FROM analytics.v_weekly_category_demand')).scalar_one()
print('PostgreSQL View 연결 성공 | 행 수:', view_count)

PostgreSQL View 연결 성공 | 행 수: 380


In [19]:
# TASK B-08-2. 주간 수요 데이터 로딩

# analytics.v_weekly_category_demand 전체 조회 SQL 작성
query = '''
    SELECT  * 
      FROM  analytics.v_weekly_category_demand
     ORDER
        BY  product_category_name, week_start; 
'''

# pd.read_sql()로 DataFrame 생성
with engine.connect() as conn:
    df = pd.read_sql(text(query), conn)


# week_start 컬럼을 datetime 타입으로 변환
df['week_start'] = pd.to_datetime(df['week_start'])

# 데이터 확인
print('데이터 크기:', df.shape)
display(df.head())
print(df.dtypes)

데이터 크기: (380, 3)


,product_category_name,week_start,weekly_items
0,beleza_saude,2017-03-06,43
1,beleza_saude,2017-03-13,52
2,beleza_saude,2017-03-20,56
3,beleza_saude,2017-03-27,44
4,beleza_saude,2017-04-03,37


product_category_name              str
week_start               datetime64[s]
weekly_items                     int64
dtype: object


In [20]:
# TASK B-08-3. 학습·평가 데이터 분리

# 상품군, 주차 순으로 오름차순 정렬
df_sorted = df.sort_values(by=['product_category_name', 'week_start'],
    ascending=[True, True]).copy()

# 학습 종료일 지정
train_end = pd.Timestamp('2018-05-21')

# 학습 데이터 추출
training_df = df_sorted[df_sorted['week_start'] <= train_end]

# 최종 평가 데이터 추출
test_df = df_sorted[df_sorted['week_start'] > train_end]

# 분리 결과 확인
print('학습:', len(training_df))
print('평가:', len(test_df))

학습: 320
평가: 60


In [26]:
# TASK B-08-4. Naive Forecast 단일 예측

# cama_mesa_banho 상품군만 추출
category_df = training_df[
    training_df.product_category_name == 'cama_mesa_banho'
].copy()

# 주차 오름차순 정렬
category_df = category_df.sort_values(by='week_start', ascending=True).copy()

# 최초 60주를 history로 추출
history = category_df.iloc[:-4]

# 마지막 4주를 actual로 추출
actual = category_df.iloc[-4:]

# history의 마지막 관측값 추출
last_value = history.iloc[-1].weekly_items

# last_value를 4번 반복해 예측값 생성
pred = np.repeat(last_value, 4)

#검증
print('History:', len(history))
print('Actual:', len(actual))
print('마지막 관측 주차:', history.iloc[-1].week_start)
print('예측 기준값:', last_value)
print('예측값:', pred)
print('실제값:', actual.weekly_items.to_numpy())

History: 60
Actual: 4
마지막 관측 주차: 2018-04-23 00:00:00
예측 기준값: 161
예측값: [161 161 161 161]
실제값: [187 164 218 133]


In [ ]:
def naive_forecast(history, horizon=4):
    last_value = history.iloc[-1]['weekly_items']
    pred = np.repeat(last_value, horizon)
    return pred
    
def wape(y_true, y_pred):
    # 실제값과 예측값의 절대오차 합계
    numerator = np.abs(y_true - y_pred).sum()

    # 실제값의 절댓값 합계
    denominator = np.abs(y_true).sum()

    # denominator가 0인지 확인
    if denominator == 0:
        raise ValueError("WAPE를 계산할 수 없습니다: 실제값의 절댓값 합계가 0입니다.")

    return numerator / denominator * 100
    
pred = naive_forecast(history, 4)
wape_value = wape(actual.weekly_items, pred)
mae_value = mean_absolute_error(actual.weekly_items, pred)

print('예측값:', pred)
print('실제값:', actual['weekly_items'].to_numpy())
print(f'MAE: {mae_value:.2f}')
print(f'WAPE: {wape_value:.2f}%')

예측값: [161 161 161 161]
실제값: [187 164 218 133]
MAE: 28.50
WAPE: 16.24%


In [37]:
# TASK B-08-5. Rolling-origin 검증

# 1. 최초 학습 기간: 32주
min_history = 32

# 2. 예측 기간: 4주
horizon = 4
result = []
# 예측 시작점을 반복하는 for문 작성
for start_point in range(min_history, len(category_df) - horizon + 1 ):
     
# 각 시점까지의 history 추출
    training = category_df.iloc[:start_point]
# 그다음 4주의 actual 추출
    actual = category_df[start_point : start_point + horizon]
# naive_forecast()로 pred 계산
    pred = naive_forecast(training, 4)
# 예측 기준일, 실제값, 예측값 저장
    for h in range(1, horizon + 1):
        result.append({
            'origin_date': training.iloc[-1]['week_start'],
            'target_date': actual.iloc[h-1]['week_start'],
            'horizon': h,
            'y_true': actual.iloc[h-1]['weekly_items'],
            'y_pred': pred[h-1]
    })
result_df = pd.DataFrame(result)

print('결과 행 수:', len(result_df))
print('예측 시작점 수:', result_df['origin_date'].nunique())
display(result_df.head(8))

결과 행 수: 116
예측 시작점 수: 29


,origin_date,target_date,horizon,y_true,y_pred
0,2017-10-09,2017-10-16,1,142,121
1,2017-10-09,2017-10-23,2,132,121
2,2017-10-09,2017-10-30,3,126,121
3,2017-10-09,2017-11-06,4,155,121
4,2017-10-16,2017-10-23,1,132,142
5,2017-10-16,2017-10-30,2,126,142
6,2017-10-16,2017-11-06,3,155,142
7,2017-10-16,2017-11-13,4,137,142


In [38]:
# TASK B-08-6. Horizon별 예측 성능 평가

metrics = []

# horizon별로 그룹화
for h, group in result_df.groupby('horizon'):

    # MAE 계산
    mae_value = mean_absolute_error(group['y_true'], group['y_pred'])

    # WAPE 계산
    wape_value = wape(group['y_true'], group['y_pred'])

    # 결과 저장
    metrics.append({
        'horizon': h,
        'n_predictions': len(group),
        'MAE': mae_value,
        'WAPE': wape_value
    })

# DataFrame 변환
metrics_df = pd.DataFrame(metrics)

display(metrics_df)

,horizon,n_predictions,MAE,WAPE
0,1,29,38.482759,21.985816
1,2,29,47.793103,27.187132
2,3,29,52.758621,29.513889
3,4,29,56.931034,31.805047


In [51]:
# TASK B-08-7. 전체 상품군 Rolling-origin 검증

min_history = 32
horizon = 4
all_results = []

# training_df를 상품군별로 그룹화
for category, group in training_df.groupby('product_category_name'):

    # 주차 오름차순으로 정렬
    category_df = group.sort_values('week_start', ascending=True).copy()

    # 기존 Rolling-origin 반복문
    for start_point in range(min_history, len(category_df) - horizon + 1 ):

        training = category_df.iloc[:start_point]
        actual = category_df[start_point : start_point+horizon]
        pred = naive_forecast(training, horizon)

        # Horizon별 예측 결과 저장
        for h in range(1, horizon + 1):
            all_results.append({
                'category': category,
                'origin_date': training.iloc[-1]['week_start'],
                'target_date': actual.iloc[h-1]['week_start'],
                'horizon': h,
                'y_true': actual.iloc[h-1]['weekly_items'],
                'y_pred': pred[h-1]
            })

# DataFrame으로 변환
all_results_df = pd.DataFrame(all_results)

print('전체 결과:', all_results_df.shape)

display(
    all_results_df.groupby(['category', 'horizon'])
    .size()
    .reset_index(name='n_predictions')
)

전체 결과: (580, 6)


,category,horizon,n_predictions
0,beleza_saude,1,29
1,beleza_saude,2,29
2,beleza_saude,3,29
3,beleza_saude,4,29
4,cama_mesa_banho,1,29
5,cama_mesa_banho,2,29
6,cama_mesa_banho,3,29
7,cama_mesa_banho,4,29
8,esporte_lazer,1,29
9,esporte_lazer,2,29


In [52]:
# TASK B-08-8. 상품군별·Horizon별 예측 성능 평가

all_metrics = []

# category와 horizon 두 컬럼으로 그룹화
for (category, h), group in all_results_df.groupby(['category', 'horizon']):

    # MAE 계산
    mae_value = mean_absolute_error(group.y_true, group.y_pred)

    # WAPE 계산
    wape_value = wape(group.y_true, group.y_pred)

    # 결과 딕셔너리를 all_metrics에 추가
    all_metrics.append({
            'category' : category,
            'horizon': h,
            'n_predictions': len(group),
            'MAE': mae_value,
            'WAPE': wape_value
        }) 
# DataFrame 생성
all_metrics_df = pd.DataFrame(all_metrics)

all_metrics_df.shape
display(all_metrics_df)

,category,horizon,n_predictions,MAE,WAPE
0,beleza_saude,1,29,27.931034,19.636364
1,beleza_saude,2,29,41.758621,28.413890
2,beleza_saude,3,29,48.034483,32.052462
3,beleza_saude,4,29,53.896552,35.684932
4,cama_mesa_banho,1,29,38.482759,21.985816
5,cama_mesa_banho,2,29,47.793103,27.187132
6,cama_mesa_banho,3,29,52.758621,29.513889
7,cama_mesa_banho,4,29,56.931034,31.805047
8,esporte_lazer,1,29,24.724138,17.428294
9,esporte_lazer,2,29,34.827586,24.343215


### Naive Forecast 검증 결과

5개 상품군을 대상으로 최초 32주부터 학습 구간을 확장하는 Rolling-origin 검증을 수행하였다. 각 상품군에 대해 29개 예측 시작점에서 향후 1~4주 수요를 예측하였다.

검증 결과 모든 상품군에서 4주 후 WAPE가 1주 후보다 높았으며, 특히 `informatica_acessorios`는 25.80%에서 45.31%로 증가하였다. 이는 마지막 관측값을 그대로 유지하는 Naive Forecast가 중장기 수요 변화를 충분히 반영하지 못할 가능성을 보여준다.

상품군별 성능에도 차이가 나타났다. 따라서 이후 Moving Average, ETS, ARIMA 모델이 단순 예측보다 얼마나 개선되는지 상품군별·예측 시차별로 비교할 필요가 있다.

본 결과는 학습 기간 내부 검증에 해당하며, 최종 12주 평가 데이터는 모델 선정 이후 사용한다.

In [53]:
# TASK B-08-9. Moving Average Forecast

def moving_average_forecast(history, horizon=4, window=4):
    # history에서 최근 window주 주문 수량 추출
    recent =  history['weekly_items'].iloc[-window:]
    # 해당 수량의 평균 계산
    recent_mean = recent.mean()
    # 평균값을 horizon만큼 반복하여 반환   
    pred = np.repeat(recent_mean, horizon)
    return pred 

pred_ma = moving_average_forecast(history, horizon=4, window=4)

print('최근 4주 실제값:', history['weekly_items'].tail(4).to_numpy())
print('이동평균 예측값:', pred_ma)

최근 4주 실제값: [174 183 181 161]
이동평균 예측값: [174.75 174.75 174.75 174.75]


In [54]:
# TASK B-08-10. Moving Average Rolling-origin 검증

min_history = 32
horizon = 4
ma_results = []

# training_df를 상품군별로 그룹화
for category, group in training_df.groupby('product_category_name'):

    # 주차 오름차순으로 정렬
    category_df = group.sort_values('week_start', ascending=True).copy()

    # 기존 Rolling-origin 반복문
    for start_point in range(min_history, len(category_df) - horizon + 1 ):

        training = category_df.iloc[:start_point]
        actual = category_df[start_point : start_point+horizon]
        pred = moving_average_forecast(training, horizon)

        # Horizon별 예측 결과 저장
        for h in range(1, horizon + 1):
            ma_results.append({
                'category': category,
                'origin_date': training.iloc[-1]['week_start'],
                'target_date': actual.iloc[h-1]['week_start'],
                'horizon': h,
                'y_true': actual.iloc[h-1]['weekly_items'],
                'y_pred': pred[h-1]
            })

# DataFrame으로 변환
ma_results_df = pd.DataFrame(ma_results)

print('Moving Average 결과:', ma_results_df.shape)

print('Naive 결과:', all_results_df.shape)

keys = ['category', 'origin_date', 'target_date', 'horizon', 'y_true']

assert ma_results_df[keys].reset_index(drop=True).equals(
    all_results_df[keys].reset_index(drop=True)
)

print('동일한 평가 데이터 사용: PASS')

Moving Average 결과: (580, 6)
Naive 결과: (580, 6)
동일한 평가 데이터 사용: PASS


In [55]:
# TASK B-08-11. Moving Average 성능 평가

ma_metrics = []

# 상품군과 Horizon별 그룹화
for (category, h), group in ma_results_df.groupby(['category', 'horizon']):

    # MAE 계산
    mae_value = mean_absolute_error(group.y_true, group.y_pred)

    # WAPE 계산
    wape_value = wape(group.y_true, group.y_pred)

    # 결과 저장
    ma_metrics.append({
        'category': category,
        'horizon': h,
        'n_predictions': len(group),
        'MAE': mae_value,
        'WAPE': wape_value
    })

# DataFrame 생성
ma_metrics_df = pd.DataFrame(ma_metrics)

display(ma_metrics_df)

,category,horizon,n_predictions,MAE,WAPE
0,beleza_saude,1,29,34.189655,24.036364
1,beleza_saude,2,29,41.724138,28.390427
2,beleza_saude,3,29,43.103448,28.762080
3,beleza_saude,4,29,44.948276,29.760274
4,cama_mesa_banho,1,29,42.224138,24.123325
5,cama_mesa_banho,2,29,49.051724,27.903099
6,cama_mesa_banho,3,29,50.534483,28.269676
7,cama_mesa_banho,4,29,50.637931,28.289347
8,esporte_lazer,1,29,28.689655,20.223627
9,esporte_lazer,2,29,35.862069,25.066281


In [61]:
# # TASK B-08-12. Naive vs MA4 비교

# 두 성능표를 category, horizon 기준으로 병합
comparison_df = ma_metrics_df.merge(
    all_metrics_df,
    on=['category', 'horizon'],
    suffixes=('_ma4', '_naive')
)

# WAPE 차이 계산 (단위: %p)
comparison_df['WAPE_diff_pp'] =  comparison_df.WAPE_naive - comparison_df.WAPE_ma4

# Naive 대비 WAPE 상대 개선율 계산 (%)
comparison_df['improvement_pct'] = (comparison_df.WAPE_diff_pp 
                                    / comparison_df.WAPE_naive *100)

display(
    comparison_df[
        ['category', 'horizon', 'WAPE_naive',
         'WAPE_ma4', 'WAPE_diff_pp', 'improvement_pct']
    ].round(2)
)

,category,horizon,WAPE_naive,WAPE_ma4,WAPE_diff_pp,improvement_pct
0,beleza_saude,1,19.64,24.04,-4.40,-22.41
1,beleza_saude,2,28.41,28.39,0.02,0.08
2,beleza_saude,3,32.05,28.76,3.29,10.27
3,beleza_saude,4,35.68,29.76,5.92,16.60
4,cama_mesa_banho,1,21.99,24.12,-2.14,-9.72
5,cama_mesa_banho,2,27.19,27.90,-0.72,-2.63
6,cama_mesa_banho,3,29.51,28.27,1.24,4.22
7,cama_mesa_banho,4,31.81,28.29,3.52,11.05
8,esporte_lazer,1,17.43,20.22,-2.80,-16.04
9,esporte_lazer,2,24.34,25.07,-0.72,-2.97


## B-08. 기준 예측 모델 구축 및 성능 비교 결과

### 1. 분석 목적

상품군별 주간 주문 수량 예측을 위해 Naive Forecast와 4주 이동평균(MA4) 모델을 구축하였다.

두 모델의 예측 성능을 비교하여, 이후 적용할 ETS·ARIMA 모델이 실제로 예측 정확도를 개선하는지 판단하기 위한 기준 성능을 확보하고자 하였다.

### 2. 분석 및 검증 방법

**데이터 구성**

- 분석 대상: 주문 상품 수량 기준 상위 5개 상품군
- 전체 관측 기간: 76주
- 학습 및 모델 검증: 최초 64주
- 최종 평가: 마지막 12주 (현재 단계에서는 사용하지 않음)

**예측 모델**

- **Naive Forecast:** 마지막 관측 주차의 주문 수량을 향후 1~4주 예측값으로 사용
- **MA4:** 최근 4주의 평균 주문 수량을 향후 1~4주 예측값으로 사용

**검증 방법**

Expanding-window 방식의 Rolling-origin 검증을 수행하였다.

- 최초 학습 기간: 32주
- 예측 시점 이동 간격: 1주
- 예측 기간: 1~4주
- 상품군별 예측 시작점: 29개
- 전체 예측 결과: 580개

모든 모델은 동일한 예측 시작점과 평가 구간을 사용하였다.

평가 지표는 평균 절대오차(MAE)와 실제 수요 규모 대비 절대오차를 나타내는 WAPE를 사용하였다.

### 3. 주요 분석 결과

#### 3.1. 상품군별 예측 성능 비교

아래는 Naive와 MA4의 1주 및 4주 후 WAPE를 비교한 결과이다.

| 상품군 | 1주 Naive | 1주 MA4 | 4주 Naive | 4주 MA4 |
|---|---:|---:|---:|---:|
| beleza_saude | 19.64% | 24.04% | 35.68% | 29.76% |
| cama_mesa_banho | 21.99% | 24.12% | 31.81% | 28.29% |
| esporte_lazer | 17.43% | 20.22% | 31.89% | 28.69% |
| informatica_acessorios | 25.80% | 33.11% | 45.31% | 48.15% |
| moveis_decoracao | 29.75% | 35.62% | 44.46% | 33.39% |

#### 3.2. 주요 발견

**첫째, 단기 예측에서는 Naive가 우수하였다.**

1주 후 예측에서 모든 상품군의 Naive WAPE가 MA4보다 낮았다. 이는 최근 관측값 하나가 단기 수요 예측에 유용한 정보를 제공할 수 있음을 보여준다.

**둘째, 예측 시차가 길어지면 MA4가 상대적으로 유리해지는 경향이 나타났다.**

4주 후 예측에서는 5개 상품군 중 4개에서 MA4의 성능이 더 우수하였다.

특히 `moveis_decoracao`는 4주 후 WAPE가 44.46%에서 33.39%로 감소하여, Naive 대비 상대 개선율 24.90%를 기록하였다.

**셋째, 모든 상품군에서 이동평균이 효과적인 것은 아니었다.**

`informatica_acessorios`는 1~4주 모든 예측 시차에서 Naive가 더 우수하였다. 이는 상품군마다 수요 변화의 특성이 다르며, 하나의 기준 예측 모델을 일괄 적용하는 데 한계가 있음을 보여준다.

### 4. 분석의 한계

- 본 결과는 학습 구간 내부 검증 성능이며, 최종 12주 테스트 성능을 의미하지 않는다.
- Rolling-origin 검증의 평가 기간이 중첩되므로 예측 오차를 모두 독립적인 관측치로 해석할 수 없다.
- Naive와 MA4는 추세를 직접 모형화하지 않으므로 지속적인 수요 증가·감소에 대응하는 데 한계가 있다.
- 블랙프라이데이로 추정되는 수요 급증 구간 등 특수한 변동이 전체 평가 결과에 영향을 미쳤을 가능성이 있다.
- 이번 비교에서는 이동평균의 기간을 4주로 고정했으며, 다른 기간의 이동평균은 검증하지 않았다.

### 5. 결론 및 다음 분석 방향

기준 모델 비교 결과, **예측 시차와 상품군에 따라 유리한 예측 방법이 달라지는 것으로 나타났다.**

특히 단기 예측에서는 최근 관측값을 사용하는 Naive가, 상대적으로 긴 예측 기간에서는 최근 여러 주를 평활한 MA4가 유리한 경우가 많았다.

따라서 다음 단계에서는 수준 및 추세 변화를 반영할 수 있는 ETS와 자기상관 구조를 활용하는 ARIMA를 적용하고, 동일한 Rolling-origin 검증 조건에서 기준 모델 대비 성능 개선 여부를 평가한다.

이후 모델 선정이 완료되면 마지막 12주를 활용하여 최종 예측 성능을 검증한다.